# 30 · Live audio and bidirectional sessions

LiveRequestQueue sends input while Runner.run_live receives events. Flash-Lite is not a Live audio model: supply an available Vertex Live model explicitly to run this optional exercise.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** No default call; optional Live model audio billing. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.agents import LiveRequestQueue, RunConfig
from google.adk.agents.run_config import StreamingMode
import asyncio, wave

async def live_demo(model_id):
    agent = llm("live_shop", "Answer in one short sentence. You help with ecommerce orders.", model=gemini(model_id))
    lab = await make_lab(agent)
    queue = LiveRequestQueue()
    queue.send_content(types.Content(role="user", parts=[types.Part(text="Say hello to a customer checking an invoice.")]))
    chunks = []
    stream = lab.runner.run_live(user_id=lab.user_id, session_id=lab.session.id,
        live_request_queue=queue,
        run_config=RunConfig(streaming_mode=StreamingMode.BIDI, response_modalities=["AUDIO"],
                             output_audio_transcription=types.AudioTranscriptionConfig()))
    try:
        async with asyncio.timeout(30):
            async for event in stream:
                if event.output_transcription:
                    print(event.output_transcription.text or "", end="")
                if event.content:
                    for part in event.content.parts or []:
                        if part.inline_data and part.inline_data.mime_type.startswith("audio/pcm"):
                            chunks.append(part.inline_data.data)
                if event.turn_complete:
                    break
    finally:
        queue.close()
        await stream.aclose()
    if chunks:
        target = ROOT / ".runtime" / "live-reply.wav"
        target.parent.mkdir(exist_ok=True)
        with wave.open(str(target), "wb") as wav:
            wav.setnchannels(1)
            wav.setsampwidth(2)
            wav.setframerate(24000)  # Gemini Live output PCM rate; verify for your model.
            wav.writeframes(b"".join(chunks))
        print("\nSaved", target)

if CONFIG["enable_optional_cloud_demos"] and CONFIG["live_model"]:
    await live_demo(CONFIG["live_model"])
else:
    print("Set live_model and enable_optional_cloud_demos in config.json to run a 30-second bounded demo.")


For microphone input, send 16 kHz mono PCM chunks with `queue.send_realtime(types.Blob(data=pcm_bytes, mime_type='audio/pcm;rate=16000'))`. Input capture belongs to your browser/audio application. A corporate proxy may also need WebSocket support; disabling HTTP certificate checks cannot fix a blocked WebSocket route.


## Try it

Choose a Live model and supported location from Google's model documentation. Compare output transcription events with audio parts.


## Reference

[Official ADK documentation](https://adk.dev/streaming/). Shared configuration: `config.json`. No environment activation or login cells are needed.
